# Hari 12-14 — Scaling, Time-Series Split, dan Penutupan Minggu 2

Lanjutan dari `dataset_siap_modeling.csv` hasil Hari 10-11 (setelah perbaikan bug kemarin).

**Catatan penyesuaian urutan:** roadmap awal menaruh "scaling" di Hari 12 sebelum "split" di Hari 13. Setelah dipikir ulang, urutan itu sebenarnya **berisiko data leakage** — kalau scaler di-fit ke seluruh data (termasuk yang nanti jadi test set), statistik dari test set diam-diam "bocor" ke proses training. Jadi di notebook ini: **Hari 12** membahas konsepnya dulu, **Hari 13** baru benar-benar split + fit scaler (dengan urutan yang benar: split dulu, baru fit scaler HANYA dari data train), **Hari 14** konsolidasi penutup Minggu 2.

---
# BAGIAN A — Hari 12: Kenapa & Kapan Perlu Scaling

In [13]:
import pandas as pd
import numpy as np

df_fitur = pd.read_csv("dataset_siap_modeling.csv", index_col=0, parse_dates=True)
print(f"Data dimuat: {df_fitur.shape[0]} baris, {df_fitur.shape[1]} kolom")
df_fitur.describe()

Data dimuat: 143 baris, 18 kolom


,lag_1,lag_2,lag_3,rolling_mean_4w,vaksin_persen,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9,bulan_10,bulan_11,bulan_12,target_minggu_depan
count,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000,143.000000
mean,46790.571096,46661.368298,46451.906760,46691.803613,32.868379,0.069930,0.055944,0.062937,0.083916,0.104895,0.083916,0.090909,0.097902,0.083916,0.097902,0.090909,0.076923,46903.620047
std,73065.363500,73131.893033,73222.906677,67794.755272,33.156741,0.255926,0.230621,0.243703,0.278236,0.307495,0.278236,0.288490,0.298227,0.278236,0.298227,0.288490,0.267406,72999.985694
min,397.000000,111.000000,6.000000,321.250000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,988.000000
25%,7376.500000,7128.500000,6958.500000,7442.875000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7600.500000
50%,24932.000000,24932.000000,24273.000000,26311.500000,18.688826,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,24932.000000
75%,41703.500000,41703.500000,41703.500000,43163.583333,73.219747,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,41703.500000
max,389727.000000,389727.000000,389727.000000,308061.250000,76.020068,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,389727.000000


Perhatikan hasil `.describe()` di atas — rentang nilai antar kolom **jauh berbeda**: `lag_1`/`lag_2`/`lag_3`/`rolling_mean_4w` bisa puluhan ribu, `vaksin_persen` cuma 0-100, sementara kolom `bulan_*` cuma berisi 0 atau 1. Ini masalah untuk model yang sensitif terhadap skala.

**Model yang butuh scaling:** Linear Regression (terutama versi ber-regularisasi seperti Ridge/Lasso), KNN, SVM, Neural Network — semuanya menghitung jarak atau memberi bobot yang bisa didominasi fitur berskala besar kalau tidak disamakan dulu.

**Model yang TIDAK butuh scaling:** Decision Tree, Random Forest, Gradient Boosting — model berbasis pohon membuat keputusan lewat threshold per fitur satu-satu, jadi tidak peduli skala antar kolom.

**Aturan penting:** kolom hasil one-hot encoding (`bulan_1`...`bulan_12`) **sebaiknya tidak ikut di-scale** — nilainya sudah bermakna sebagai 0/1, men-scale malah merusak interpretasinya.

## Ilustrasi Kenapa Urutan Split→Scaling Itu Penting

In [14]:
# Cell ini sudah lengkap — demonstrasi kenapa fit scaler ke SELURUH data itu berisiko.

# Skenario SALAH: fit scaler ke seluruh data (termasuk yang harusnya jadi test set)
mean_semua_data = df_fitur["lag_1"].mean()

# Skenario BENAR: fit scaler HANYA ke bagian yang akan jadi train (asumsikan 80% pertama)
cutoff_ilustrasi = int(len(df_fitur) * 0.8)
mean_train_saja = df_fitur["lag_1"].iloc[:cutoff_ilustrasi].mean()

print(f"Mean lag_1 dari SELURUH data (salah)  : {mean_semua_data:,.1f}")
print(f"Mean lag_1 dari TRAIN saja (benar)     : {mean_train_saja:,.1f}")
print("\nBedanya mungkin tidak besar di kasus ini, tapi prinsipnya tetap sama:")
print("scaler yang dipakai untuk transform test set HARUS di-fit hanya dari train,")
print("supaya test set benar-benar mensimulasikan data yang belum pernah 'dilihat' model.")

Mean lag_1 dari SELURUH data (salah)  : 46,790.6
Mean lag_1 dari TRAIN saja (benar)     : 53,025.0

Bedanya mungkin tidak besar di kasus ini, tapi prinsipnya tetap sama:
scaler yang dipakai untuk transform test set HARUS di-fit hanya dari train,
supaya test set benar-benar mensimulasikan data yang belum pernah 'dilihat' model.


---
# BAGIAN B — Hari 13: Time-Series Split + Scaling yang Benar

## Kenapa Time-Series Tidak Boleh Di-split Acak

`train_test_split()` biasa akan mengacak baris sebelum membagi — untuk data time-series ini **fatal**: bisa saja minggu di tahun 2022 masuk train, sementara minggu di tahun 2021 masuk test. Model jadi "melihat masa depan" saat training, padahal di dunia nyata kita tidak akan pernah punya data masa depan saat prediksi. Aturannya: **data lama → train, data terbaru → test**, berurutan sesuai waktu.

In [15]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Pisahkan df_fitur menjadi X (semua kolom KECUALI "target_minggu_depan") dan y (kolom "target_minggu_depan" saja)
#    Hint: X = df_fitur.drop(columns=["target_minggu_depan"])
# 2. Tentukan titik potong (cutoff) untuk 80% data pertama jadi train:
#    cutoff = int(len(X) * 0.8)
# 3. Split berurutan waktu (BUKAN train_test_split biasa) pakai .iloc:
#    X_train = X.iloc[:cutoff]      X_test = X.iloc[cutoff:]
#    y_train = y.iloc[:cutoff]      y_test = y.iloc[cutoff:]
# 4. Cetak shape keempatnya, dan cetak juga tanggal awal-akhir X_train dan X_test
#    (pakai .index.min() dan .index.max()) untuk memastikan tidak ada tumpang tindih waktu antara train dan test
# Tulis kode kamu di bawah ini:
x = df_fitur.drop(columns=["target_minggu_depan"])
y = df_fitur["target_minggu_depan"]
cutoff = int(len(x) * 0.8)

X_train = x.iloc[:cutoff]
X_test = x.iloc[cutoff:]

y_train = y.iloc[:cutoff]
y_test = y.iloc[cutoff:]

print("Shape x_train:", X_train.shape)
print("Shape x_test:", X_test.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)
print("Periode x_train:", X_train.index.min(), "sampai", X_train.index.max())
print("Periode x_test:", X_test.index.min(), "sampai", X_test.index.max())

Shape x_train: (114, 17)
Shape x_test: (29, 17)
Shape y_train: (114,)
Shape y_test: (29,)
Periode x_train: 2020-03-29 00:00:00+00:00 sampai 2022-05-29 00:00:00+00:00
Periode x_test: 2022-06-05 00:00:00+00:00 sampai 2022-12-18 00:00:00+00:00


## Fit Scaler HANYA dari Train, lalu Transform Train & Test

In [16]:
# Cell ini sudah lengkap — pola standar scikit-learn: fit HANYA di train, transform di keduanya.

from sklearn.preprocessing import StandardScaler

# Kolom yang perlu di-scale (kontinu), kolom bulan_* TIDAK ikut di-scale
kolom_kontinu = ["lag_1", "lag_2", "lag_3", "rolling_mean_4w", "vaksin_persen"]
kolom_bulan = [col for col in x.columns if col.startswith("bulan_")]

scaler = StandardScaler()

# fit_transform HANYA di train
X_train_scaled_kontinu = scaler.fit_transform(X_train[kolom_kontinu])
# transform SAJA (tanpa fit ulang) di test — pakai statistik dari train
X_test_scaled_kontinu = scaler.transform(X_test[kolom_kontinu])

# Gabungkan kembali dengan kolom bulan_* yang tidak di-scale
X_train_scaled = pd.DataFrame(X_train_scaled_kontinu, columns=kolom_kontinu, index=X_train.index)
X_train_scaled = pd.concat([X_train_scaled, X_train[kolom_bulan]], axis=1)

X_test_scaled = pd.DataFrame(X_test_scaled_kontinu, columns=kolom_kontinu, index=X_test.index)
X_test_scaled = pd.concat([X_test_scaled, X_test[kolom_bulan]], axis=1)

print("Statistik lag_1 SEBELUM scaling (train):")
print(X_train["lag_1"].describe()[["mean", "std"]])
print("\nStatistik lag_1 SESUDAH scaling (train) — mean harus ~0, std harus ~1:")
print(X_train_scaled["lag_1"].describe()[["mean", "std"]])

Statistik lag_1 SEBELUM scaling (train):
mean    53025.014620
std     80461.911642
Name: lag_1, dtype: float64

Statistik lag_1 SESUDAH scaling (train) — mean harus ~0, std harus ~1:
mean    8.570143e-17
std     1.004415e+00
Name: lag_1, dtype: float64


Catatan: kamu sekarang punya **dua versi** fitur — `X_train`/`X_test` (belum di-scale, dipakai untuk model tree-based di Hari 17) dan `X_train_scaled`/`X_test_scaled` (sudah di-scale, dipakai untuk model linear di Hari 16). Simpan keduanya.

---
# BAGIAN C — Hari 14: Konsolidasi & Penutupan Minggu 2

In [17]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# Lakukan sanity check terakhir sebelum Modeling:
# 1. Cetak shape dari X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled
# 2. Cek tidak ada NaN tersisa di X_train dan X_test dengan .isnull().sum().sum()
#    (harus mengembalikan 0)
# 3. Cek tidak ada tumpang tindih waktu: pastikan X_train.index.max() < X_test.index.min()
#    (pakai assert supaya notebook langsung error kalau ternyata salah — cara ini disebut
#    "sanity check" atau "assertion", praktik umum sebelum lanjut ke tahap berikutnya)
# Tulis kode kamu di bawah ini:
print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test)
print("Shape y_train:", y_train)
print("Shape y_test:", y_test)
print("Shape X_train_scaled", X_train_scaled)
print("Shape X_test_scaled", X_test_scaled)

print("cek tidak ada Nan Shape X_train:", X_train.isnull().sum().sum())
print("cek tidak ada Nan Shape X_test:", X_test.isnull().sum().sum())

#print("cek tidak ada tumpang tindih waktu:", X_train.index.max() < X_test.index.min())
assert X_train.index.max() < X_test.index.min(), "Ada tumpang tindih waktu antara train dan test!"

Shape X_train: (114, 17)
Shape X_test:                              lag_1    lag_2    lag_3  rolling_mean_4w  \
Date                                                                    
2022-06-05 00:00:00+00:00   1825.0   1814.0   2345.0          2092.25   
2022-06-12 00:00:00+00:00   2385.0   1825.0   1814.0          2428.00   
2022-06-19 00:00:00+00:00   3688.0   2385.0   1825.0          3871.25   
2022-06-26 00:00:00+00:00   7587.0   3688.0   2385.0          6509.00   
2022-07-03 00:00:00+00:00  12376.0   7587.0   3688.0          9279.25   
2022-07-10 00:00:00+00:00  13466.0  12376.0   7587.0         12704.25   
2022-07-17 00:00:00+00:00  17388.0  13466.0  12376.0         16719.50   
2022-07-24 00:00:00+00:00  23648.0  17388.0  13466.0         21972.75   
2022-07-31 00:00:00+00:00  33389.0  23648.0  17388.0         28295.25   
2022-08-07 00:00:00+00:00  38756.0  33389.0  23648.0         33418.25   
2022-08-14 00:00:00+00:00  37880.0  38756.0  33389.0         36955.25   
2022-08-21 0

In [18]:
# Cell ini sudah lengkap — menyimpan semua set data final untuk dipakai di Minggu 3.

X_train.to_csv("X_train.csv")
X_test.to_csv("X_test.csv")
X_train_scaled.to_csv("X_train_scaled.csv")
X_test_scaled.to_csv("X_test_scaled.csv")
y_train.to_csv("y_train.csv")
y_test.to_csv("y_test.csv")

print("6 file tersimpan: X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test")

6 file tersimpan: X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test


## Data Preparation Summary Report — Minggu 2

**Proyek:** Prediksi Tren Kasus COVID-19 Mingguan Indonesia

**Penanganan Outlier (Hari 8-9)**
- 4 minggu terindikasi outlier lewat analisis residual dekomposisi time-series
- Ditangani lewat investigasi manual per-tanggal (dicocokkan dengan timeline gelombang COVID-19), sebagian dipertahankan (sinyal asli, dekat puncak gelombang), sebagian di-smoothing lokal

**Feature Engineering (Hari 10-11)**
- Fitur lag: `lag_1`, `lag_2`, `lag_3` (kasus 1-3 minggu sebelumnya)
- Fitur tren: `rolling_mean_4w` (rata-rata trailing 4 minggu)
- Fitur eksternal: `vaksin_persen` (persentase populasi tervaksin dosis-1)
- Fitur musiman: `bulan_1` s/d `bulan_12` (one-hot encoding bulan)
- Target: `target_minggu_depan` (kasus minggu berikutnya — dibuat lewat shift(-1))

**Split & Scaling (Hari 12-13)**
- Split berurutan waktu (80% data lama = train, 20% data terbaru = test) — bukan acak
- Scaler di-fit HANYA dari train untuk menghindari data leakage
- Dua versi fitur disiapkan: mentah (untuk tree-based) dan scaled (untuk linear model)

**Siap untuk Minggu 3 (Modeling):**
- Baseline pembanding (dari Hari 2): naive forecast MAE — model HARUS mengalahkan angka ini
- 6 file siap pakai: `X_train(.csv)`, `X_test`, `X_train_scaled`, `X_test_scaled`, `y_train`, `y_test`

## Refleksi Hari 14

> 1. Berapa baris data yang masuk train, dan berapa yang masuk test? → 114 dan 29
> 2. Lihat kembali baseline naive forecast MAE dari Hari 2 — model machine learning kamu di Minggu 3 harus mencetak MAE di bawah angka itu. Tuliskan lagi angkanya di sini supaya gampang dicek nanti → 12,953 kasus/minggu

---
### Selanjutnya: Minggu 3 (Hari 15-21) — Modeling

- **Hari 15**: Konsep dasar ML — regresi vs klasifikasi (proyekmu masuk regresi/forecasting)
- **Hari 16**: Linear Regression pakai `X_train_scaled` — model pertama untuk dibandingkan dengan baseline naive
- **Hari 17**: Random Forest Regressor pakai `X_train` (versi tidak di-scale, karena tree-based tidak butuh scaling)
- **Hari 18**: Evaluasi kedua model dengan MAE/RMSE, bandingkan dengan baseline Hari 2
- **Hari 19-20**: Cross-validation & tuning (catatan: cross-validation biasa juga harus disesuaikan untuk time-series — akan dibahas pakai `TimeSeriesSplit` dari scikit-learn, bukan `KFold` biasa)
- **Hari 21**: Mini-project — bandingkan semua model, pilih yang terbaik